# 02 · Wound regions and neoblast response

After amputation, planarians rebuild missing tissue from neoblasts, their adult stem cells. If neoblasts drive regeneration, they should accumulate near the cut surfaces in the first days after injury and then decline as the new tissue differentiates. This notebook tests that by locating the wound regions in each Visium section and comparing neoblast abundance near and away from the wound across the time course.

**Steps**

1. Load one section per timepoint
2. Locate the pharynx as an anatomical landmark
3. Define the wound regions in the 6 hpa section
4. Transfer the wound regions to the other timepoints
5. Compare neoblast abundance near and away from the wound
6. Follow neoblast abundance at the wound over time
7. Save the wound regions

**Input:** `data/Scanpy_Visium/Visium_*.h5ad` and `data/smed_blast.csv`.
**Output:** `data/processed/02_wound_regions.csv` (one row per spot: section, coordinates, wound label). The wound regions defined here are used throughout `03_analysis.ipynb`.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy import stats
from scipy.spatial import cKDTree

sc.settings.verbosity = 0

DATA_DIR = '../data/Scanpy_Visium'
OUT_DIR = '../data/processed'
FIG_DIR = '../figures'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

blast = pd.read_csv('../data/smed_blast.csv').dropna(subset=['gene_symbol'])
blast = blast.drop_duplicates('smed_id')
sym2ids = blast.groupby('gene_symbol')['smed_id'].apply(list).to_dict()

TP_INFO = {
    '0hpa':  ('Visium_0hpa.h5ad',  0),
    '6hpa':  ('Visium_6hpa.h5ad',  6),
    '12hpa': ('Visium_12hpa.h5ad', 12),
    '1dpa':  ('Visium_1dpa.h5ad',  24),
    '3dpa':  ('Visium_3dpa.h5ad',  72),
    '7dpa':  ('Visium_7dpa.h5ad',  168),
}
TICK_LBL = ['0h', '6h', '12h', '1d', '3d', '7d']

## 1. Load one section per timepoint

**What.** Each timepoint file holds several tissue sections cut from different depths of the animal. For each timepoint I keep only the section with the most spots.

**Why one section.** The wound regions are defined by position within a section, and the sections on a slide sit at different places and depths. Working within one section keeps the geometry simple and comparable. The largest section gives the most spots and the most complete outline of the body.

**Coordinates.** `obsm['spatial']` is in pixels of the original full-resolution image, and the image resolution differs between slides. Multiplying by each slide's `tissue_lowres_scalef` puts every section in the same units (pixels of the low-resolution image). To check that these units really match across slides, I convert them to micrometers using the Visium spot grid, where neighboring spots are always 100 µm apart.

In [ ]:
def load_max_section(fname):
    # Load the section with the most spots; attach scaled spatial coords
    a = sc.read_h5ad(os.path.join(DATA_DIR, fname))
    sec_id = a.obs['Section_id'].value_counts().idxmax()
    sec = a[a.obs['Section_id'] == sec_id].copy()
    lib_id = sec.obs['library_id'].iloc[0]
    sf = a.uns['spatial'][lib_id]['scalefactors']['tissue_lowres_scalef']
    sec.obsm['spatial_scaled'] = sec.obsm['spatial'] * sf
    del a
    return sec, sf, sec_id


sections = {}
rows = []
for tp, (fname, hours) in TP_INFO.items():
    sec, sf, sec_id = load_max_section(fname)
    sections[tp] = (sec, sf, hours)

    # Neighboring Visium spots are 100 µm apart -> micrometers per scaled pixel
    d, _ = cKDTree(sec.obsm['spatial_scaled']).query(sec.obsm['spatial_scaled'], k=2)
    um_per_px = 100 / np.median(d[:, 1])
    rows.append({'timepoint': tp, 'section': sec_id, 'n_spots': sec.n_obs,
                 'scale_factor': round(sf, 5), 'um_per_scaled_px': round(um_per_px, 1)})

print(pd.DataFrame(rows).to_string(index=False))
UM_PER_PX = np.mean([r['um_per_scaled_px'] for r in rows])

**Result.** PLACEHOLDER_S1

## 2. Locate the pharynx as an anatomical landmark

**What.** Score every spot by the summed raw counts of FOXA1 orthologs, label the top 10% of spots (with non-zero counts) as pharynx, and take their centroid as the pharynx center.

**Why.** The cut positions are not recorded in the data, so they have to be reconstructed from anatomy. The pharynx sits in the middle of the trunk fragment and is present at every timepoint, which makes it a stable reference point. FOXA1 (*foxA* in planarians) is the master regulator of pharynx identity and is expressed throughout the organ, so it marks the pharynx clearly.

**Parameters.** The top-10% cutoff picks a compact group of the most strongly FOXA1-positive spots, so the center is not pulled by scattered low-level expression elsewhere.

In [ ]:
def foxa_score(adata):
    # Sum raw counts of all FOXA1 ortholog genes per spot
    foxa_ids = [g for g in sym2ids.get('FOXA1', []) if g in adata.var_names]
    X = adata[:, foxa_ids].X
    if sp.issparse(X):
        X = X.toarray()
    return np.asarray(X).sum(axis=1).flatten()


fig, axes = plt.subplots(2, 3, figsize=(15, 9))
pharynx_centers = {}

for ax, (tp, (sec, sf, hours)) in zip(axes.flatten(), sections.items()):
    coords = sec.obsm['spatial_scaled']
    score = foxa_score(sec)

    threshold = np.percentile(score, 90)
    is_pharynx = (score >= threshold) & (score > 0)
    center = coords[is_pharynx].mean(axis=0)
    pharynx_centers[tp] = center

    im = ax.scatter(coords[:, 0], coords[:, 1], c=score, cmap='YlOrRd', s=2,
                    vmin=0, vmax=np.percentile(score, 98))
    ax.scatter(coords[is_pharynx, 0], coords[is_pharynx, 1],
               edgecolors='dodgerblue', facecolors='none', s=10, linewidths=0.7)
    ax.scatter(*center, marker='*', color='dodgerblue', s=140, zorder=5)
    ax.set_title(f'{tp}  (n={sec.n_obs})', fontsize=11)
    ax.set_aspect('equal')
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.6, label='FOXA1 counts')

plt.suptitle('Pharynx located by FOXA1 expression\n'
             'Blue outline = top 10% FOXA1 spots   ★ = pharynx center', fontsize=13)
plt.tight_layout()
plt.show()

print('Pharynx centers (scaled px):')
for tp, c in pharynx_centers.items():
    print(f'  {tp}: x={c[0]:.1f}, y={c[1]:.1f}')

**Result.** PLACEHOLDER_S2

## 3. Define the wound regions in the 6 hpa section

**What.** In the 6 hpa section, take the two ends of the tissue along the body axis (x) as the two cut surfaces. Measure the distance from the pharynx center to each cut (D_post and D_ant), and label spots within a fixed distance (the buffer) of either cut as wound-proximal.

**Why 6 hpa.** At 6 hours the fragment has just been cut and has not yet started to regrow, so the tissue edges are still the cut surfaces. At later timepoints new tissue grows out from the wound and the edges move. The two distances measured here are therefore used as the reference for all timepoints.

**Parameters.** Buffer = 30 scaled pixels on each side, a little over one anterior–posterior (AP) section in width. This is wide enough to include the wound epidermis and the tissue just behind it, where neoblasts gather.

In [ ]:
sec_6h, sf_6h, _ = sections['6hpa']
coords_6h = sec_6h.obsm['spatial_scaled']
x6 = coords_6h[:, 0]
pharynx_x6, pharynx_y6 = pharynx_centers['6hpa']

# Two cut sites: leftmost (posterior, AP-8) and rightmost (anterior, AP-2)
post_cut_x = x6.min()
ant_cut_x = x6.max()

# Distances from pharynx center to each cut site (scaled px)
D_post = pharynx_x6 - post_cut_x
D_ant = ant_cut_x - pharynx_x6

# Wound-proximal = within BUFFER px of either cut site
BUFFER = 30
is_wound_6h = (x6 <= post_cut_x + BUFFER) | (x6 >= ant_cut_x - BUFFER)

print(f'Pharynx center:     x = {pharynx_x6:.1f} px')
print(f'Posterior cut site: x = {post_cut_x:.1f} px  ->  D_post = {D_post:.1f} px '
      f'({D_post * UM_PER_PX / 1000:.2f} mm)')
print(f'Anterior cut site:  x = {ant_cut_x:.1f} px  ->  D_ant  = {D_ant:.1f} px '
      f'({D_ant * UM_PER_PX / 1000:.2f} mm)')
print(f'Buffer: {BUFFER} px = {BUFFER * UM_PER_PX:.0f} µm from each cut site')
print(f'Wound-proximal spots: {is_wound_6h.sum()} / {len(is_wound_6h)} '
      f'({100 * is_wound_6h.mean():.1f}%)')

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(coords_6h[~is_wound_6h, 0], coords_6h[~is_wound_6h, 1],
           c='lightgray', s=3, label='Wound-distal')
ax.scatter(coords_6h[is_wound_6h, 0], coords_6h[is_wound_6h, 1],
           c='crimson', s=5, label=f'Wound-proximal (within {BUFFER} px of cut)')
ax.scatter(pharynx_x6, pharynx_y6, marker='*', color='dodgerblue',
           s=200, zorder=6, label='Pharynx center')

for x_cut, x_buf in [(post_cut_x, post_cut_x + BUFFER), (ant_cut_x, ant_cut_x - BUFFER)]:
    ax.axvline(x_cut, color='black', linestyle=':', linewidth=1)
    ax.axvline(x_buf, color='crimson', linestyle='--', linewidth=1)

y_mid = coords_6h[:, 1].mean()
for x_start, x_end, label, dy in [
    (post_cut_x, pharynx_x6, f'D_post = {D_post:.0f} px', 8),
    (pharynx_x6, ant_cut_x, f'D_ant = {D_ant:.0f} px', -14),
]:
    ax.annotate('', xy=(x_end, y_mid + dy), xytext=(x_start, y_mid + dy),
                arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
    ax.text((x_start + x_end) / 2, y_mid + dy + 4, label, ha='center', fontsize=8)

ax.set_title('6 hpa: wound regions defined relative to the pharynx center')
ax.set_xlabel('x (scaled px)')
ax.set_ylabel('y (scaled px)')
ax.set_aspect('equal')
ax.legend(loc='upper center', fontsize=8)
plt.tight_layout()
plt.show()

**Result.** PLACEHOLDER_S3

## 4. Transfer the wound regions to the other timepoints

**What.** In every section, place the two cut sites at D_post and D_ant from that section's pharynx center, and label spots within the buffer of each as wound-proximal.

**Why relative to the pharynx.** Sections sit at different positions on their slides, so absolute coordinates cannot be compared. Distances from the pharynx can, because the scaled coordinates have the same units in every section (Section 1). This assumes all sections lie along the same axis, without rotation.

**Orientation check.** A section can be placed on the slide mirror-imaged, with head and tail swapped. Each spot has an AP label (AP-1 = most anterior), so I check the direction of the body axis from the data: in a normally oriented section, higher AP numbers (more posterior) sit at lower x, so the correlation between AP number and mean x is negative. A positive correlation means the section is flipped, and the two distances are applied in the opposite direction.

In [ ]:
wound_masks = {}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, (tp, (sec, sf, hours)) in zip(axes.flatten(), sections.items()):
    coords = sec.obsm['spatial_scaled']
    x = coords[:, 0]
    pharynx_x = pharynx_centers[tp][0]

    # Orientation from the AP labels:
    # normal  -> higher AP number (more posterior) at lower x -> negative corr
    # flipped -> higher AP number at higher x                 -> positive corr
    ap_obs = sec.obs[['AP_Info']].copy()
    ap_obs['x'] = x
    ap_mean = ap_obs.groupby('AP_Info', observed=True)['x'].mean()
    ap_nums = ap_mean.index.astype(str).str.replace('AP-', '').astype(int)
    flipped = np.corrcoef(ap_nums, ap_mean.values)[0, 1] > 0

    if not flipped:
        post_bound = pharynx_x - D_post + BUFFER
        ant_bound = pharynx_x + D_ant - BUFFER
        is_wound = (x <= post_bound) | (x >= ant_bound)
    else:
        post_bound = pharynx_x + D_post - BUFFER
        ant_bound = pharynx_x - D_ant + BUFFER
        is_wound = (x >= post_bound) | (x <= ant_bound)

    wound_masks[tp] = is_wound

    ax.scatter(coords[~is_wound, 0], coords[~is_wound, 1], c='lightgray', s=1.5,
               label='Wound-distal')
    ax.scatter(coords[is_wound, 0], coords[is_wound, 1], c='crimson', s=3,
               label='Wound-proximal')
    ax.scatter(*pharynx_centers[tp], marker='*', color='dodgerblue', s=120,
               zorder=5, label='Pharynx')
    flip_label = '  [mirror-imaged]' if flipped else ''
    ax.set_title(f'{tp}  (wound spots = {is_wound.sum()}){flip_label}', fontsize=10)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.legend(fontsize=6, loc='lower right')

plt.suptitle('Wound regions across the time course\n'
             'Red = wound-proximal   ★ = pharynx center', fontsize=13)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/02_wound_regions.png', dpi=150, bbox_inches='tight')
plt.show()

**Result.** PLACEHOLDER_S4

## 5. Compare neoblast abundance near and away from the wound

**What.** For each spot, compute the neoblast proportion from the cell2location estimates that come with the dataset: the neoblast estimate divided by the sum of all seven cell-type estimates. Then, in each section, test whether wound-proximal spots have a higher neoblast proportion than wound-distal spots.

**Why cell2location.** Each Visium spot covers several cells, so a spot is a mixture rather than a single cell type. cell2location estimates how many cells of each type a spot contains, using a single-cell reference. Converting to proportions removes differences in how many cells a spot holds in total.

**Test.** One-sided Mann–Whitney U test (Wilcoxon rank-sum), wound-proximal > wound-distal. Proportions are bounded and skewed, so a rank-based test is used instead of a t-test.

In [ ]:
def neoblast_prop(sec):
    ct_cols = [c for c in sec.obs.columns if c.startswith('cell2location')]
    total = sec.obs[ct_cols].sum(axis=1)
    return (sec.obs['cell2location_Neoblast'] / total).values


records = []
for tp, (sec, sf, hours) in sections.items():
    neo = neoblast_prop(sec)
    mask = wound_masks[tp]
    neo_wound, neo_distal = neo[mask], neo[~mask]
    _, pval = stats.mannwhitneyu(neo_wound, neo_distal, alternative='greater')
    records.append({
        'timepoint': tp,
        'hours': hours,
        'n_wound': int(mask.sum()),
        'n_distal': int((~mask).sum()),
        'mean_neoblast_wound': round(float(neo_wound.mean()), 4),
        'mean_neoblast_distal': round(float(neo_distal.mean()), 4),
        'p_value': pval,
    })

results_df = pd.DataFrame(records).set_index('timepoint')
print(results_df.to_string())

**Result.** PLACEHOLDER_S5

## 6. Follow neoblast abundance at the wound over time

**What.** Plot the neoblast proportion in wound-proximal spots at each timepoint, and test for a monotonic trend over time with a Spearman correlation between hours after amputation and the mean wound proportion.

**Why Spearman.** There are only six timepoints, and there is no reason to expect a linear change, so a rank correlation is the simpler and safer choice.

In [ ]:
rho, pval = stats.spearmanr(results_df['hours'], results_df['mean_neoblast_wound'])
print('Spearman correlation: time vs mean neoblast proportion at the wound')
print(f'  rho     = {rho:.3f}')
print(f'  p-value = {pval:.3f}')

In [ ]:
hours_list = list(results_df['hours'])

fig, ax = plt.subplots(figsize=(8, 5))

# Individual wound spots per timepoint
rng = np.random.default_rng(0)
for tp, (sec, sf, hours) in sections.items():
    neo = neoblast_prop(sec)
    mask = wound_masks[tp]
    jitter = rng.uniform(-1.5, 1.5, mask.sum())
    ax.scatter(np.full(mask.sum(), hours) + jitter, neo[mask],
               s=4, alpha=0.3, color='steelblue', zorder=1)

# Wound vs distal means
ax.plot(hours_list, results_df['mean_neoblast_wound'], 'o-', color='steelblue',
        linewidth=2.2, markersize=8, zorder=3, label='Wound-proximal (mean)')
ax.plot(hours_list, results_df['mean_neoblast_distal'], 's--', color='gray',
        linewidth=1.5, markersize=6, zorder=2, label='Wound-distal (mean)')

for h, v in zip(hours_list, results_df['mean_neoblast_wound']):
    ax.annotate(f'{v:.3f}', (h, v), textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_xlabel('Time after amputation', fontsize=12)
ax.set_ylabel('Neoblast proportion per spot', fontsize=12)
ax.set_title('Neoblast abundance at the wound over time', fontsize=13)
ax.set_xticks(hours_list)
ax.set_xticklabels(TICK_LBL)
ax.set_xlim(-10, 178)
ax.legend(fontsize=10)
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/02_neoblast_kinetics.png', dpi=150, bbox_inches='tight')
plt.show()

**Result.** PLACEHOLDER_S6

## 7. Save the wound regions

One row per spot of the selected sections, with its timepoint, section, scaled coordinates, neoblast proportion, and wound label. The pharynx centers and reference distances are saved alongside so the regions can be rebuilt exactly.

In [ ]:
frames = []
for tp, (sec, sf, hours) in sections.items():
    xy = sec.obsm['spatial_scaled']
    frames.append(pd.DataFrame({
        'spot': sec.obs_names,
        'timepoint': tp,
        'section': sec.obs['Section_id'].astype(str).values,
        'x_scaled': xy[:, 0],
        'y_scaled': xy[:, 1],
        'neoblast_prop': neoblast_prop(sec),
        'wound_proximal': wound_masks[tp],
    }))
wound_df = pd.concat(frames, ignore_index=True)
wound_df.to_csv(f'{OUT_DIR}/02_wound_regions.csv', index=False)

params = pd.DataFrame({
    'timepoint': list(pharynx_centers),
    'pharynx_x': [c[0] for c in pharynx_centers.values()],
    'pharynx_y': [c[1] for c in pharynx_centers.values()],
})
params['D_post'], params['D_ant'], params['buffer'] = D_post, D_ant, BUFFER
params.to_csv(f'{OUT_DIR}/02_wound_parameters.csv', index=False)

print(wound_df.groupby('timepoint', sort=False)['wound_proximal'].agg(['sum', 'size']))

## Summary

PLACEHOLDER_SUMMARY